# 03 — LoRA Fine-tuning of Protenix

Fine-tune Protenix on competition RNA data using LoRA (Low-Rank Adaptation).

**Strategy**: Insert low-rank matrices into Pairformer and Diffusion Module attention layers.
Freeze MSA Module and Input Embedder to preserve pre-trained knowledge.

**Expected**: ~2-5M trainable parameters (vs 368M total), 20-40h training on A100.

In [ ]:
# === Colab Setup Cell ===
!pip install kaggle -q
import os
from google.colab import files
if not os.path.exists('/root/.kaggle/kaggle.json'):
    print("Please upload your kaggle.json file")
    uploaded = files.upload()
    !mkdir -p /root/.kaggle && mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json

REPO_URL = "https://github.com/YOUR_USER/3drna_cc.git"  # <-- UPDATE THIS
if not os.path.exists('/content/3drna_cc'):
    !git clone {REPO_URL} /content/3drna_cc
%cd /content/3drna_cc

from src.setup import setup_environment
setup_environment()

## Step 1: Prepare Training Data

In [ ]:
from src.data.loader import load_sequences, load_labels
from src.data.featurizer import build_all_inputs
from src.config import OUTPUT_DIR

train_seq = load_sequences(split='train')
train_labels = load_labels(split='train')
val_seq = load_sequences(split='val')
val_labels = load_labels(split='val')

print(f"Train: {len(train_seq)} targets")
print(f"Val: {len(val_seq)} targets")

# Build Protenix input JSONs for training set
train_input_dir = OUTPUT_DIR / 'train_inputs'
train_paths = build_all_inputs(train_seq, output_dir=train_input_dir, n_seeds=1)
print(f"Created {len(train_paths)} training inputs")

val_input_dir = OUTPUT_DIR / 'val_inputs'
val_paths = build_all_inputs(val_seq, output_dir=val_input_dir, n_seeds=1)
print(f"Created {len(val_paths)} validation inputs")

In [ ]:
from src.model.lora_finetune import RNATrainingDataset

train_dataset = RNATrainingDataset(
    input_json_dir=train_input_dir,
    labels_df=train_labels,
    sequences_df=train_seq,
    max_crop=384,
)

val_dataset = RNATrainingDataset(
    input_json_dir=val_input_dir,
    labels_df=val_labels,
    sequences_df=val_seq,
    max_crop=384,
)

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Val dataset: {len(val_dataset)} samples")

## Step 2: Setup LoRA & Train

In [ ]:
from src.model.lora_finetune import ProtenixLoRATrainer

trainer = ProtenixLoRATrainer(
    lora_rank=16,
    lora_alpha=32,
    lora_dropout=0.05,
    device='cuda',
)

# Setup model with LoRA adapters
trainer.setup_model()

In [ ]:
# Train
history = trainer.train(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    epochs=5,
    lr=1e-4,
    gradient_accumulation=8,
    save_dir=OUTPUT_DIR / 'checkpoints',
)

In [ ]:
# Plot training curves
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_loss'], label='Train Loss')
if history['val_loss']:
    ax.plot(history['val_loss'], label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('LoRA Fine-tuning Loss')
ax.legend()
plt.show()

## Step 3: Save Final LoRA Weights

In [ ]:
from src.config import LORA_WEIGHTS

trainer.save_lora_weights(LORA_WEIGHTS)
print(f"LoRA weights saved to {LORA_WEIGHTS}")

# Check size
import subprocess
result = subprocess.run(['du', '-sh', str(LORA_WEIGHTS)], capture_output=True, text=True)
print(f"Weight size: {result.stdout.strip()}")

## Step 4: Validate Fine-tuned Model

In [ ]:
from src.model.protenix_runner import ProtenixRunner
from src.ensemble.tm_score import best_of_n_tm_score
from src.data.loader import labels_to_coords
import pandas as pd

# Run inference with LoRA weights
runner = ProtenixRunner(lora_dir=LORA_WEIGHTS, device='cuda')
results = runner.predict_all(val_input_dir, n_seeds=5)

# Compute TM-scores
tm_scores = []
for tid, preds in results.items():
    if not preds:
        continue
    ref = labels_to_coords(val_labels, tid)
    if len(ref) == 0:
        continue
    best_tm, _ = best_of_n_tm_score(preds[:5], ref)
    tm_scores.append({'target_id': tid, 'finetuned_tm': best_tm})

ft_df = pd.DataFrame(tm_scores)
print(f"Fine-tuned mean TM: {ft_df['finetuned_tm'].mean():.4f}")